# FactLedger extractor

`load(path) -> documents, units` over raw files and nothing else. The extractor sniffs the
format from the bytes and writes document and unit JSON in the shapes of SCHEMA.md; the
rules it follows are in BUILD.md. Built one block at a time. Inputs: the public raw dataset
and the private papers dataset, both attached to this notebook.


In [ ]:
# Block 1: inputs and integrity.
# Mount both datasets, count files per folder, and check every file's sha256 against the
# folder manifest. The manifests are used here only to prove the Kaggle copies are the bytes
# that were uploaded; the extractor itself never reads them.
import hashlib
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

def mount(slug):
    """Kaggle mounts inputs at /kaggle/input/<slug> or, in newer sessions,
    /kaggle/input/datasets/<owner>/<slug>. Take whichever exists."""
    for candidate in (Path("/kaggle/input") / slug, Path("/kaggle/input/datasets/jhffmn") / slug):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(slug)


RAW = mount("it494-narrative-corpora-raw")
PAPERS = mount("it494-reference-papers")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def check(folder, rows_key):
    manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
    rows = manifest[rows_key]
    on_disk = {p.name for p in folder.iterdir() if p.name not in ("manifest.json", "LICENSE")}
    listed = {r["file"] for r in rows}
    # Kaggle inputs are a network filesystem: one file at a time, 19,206 files take tens of
    # minutes; 32 concurrent reads take about a minute.
    with ThreadPoolExecutor(max_workers=32) as pool:
        digests = list(pool.map(sha256, [folder / r["file"] for r in rows]))
    bad = [r["file"] for r, d in zip(rows, digests) if d != r["sha256"]]
    print(f"{folder.name:<28} files {len(on_disk):>6}  listed {len(listed):>6}"
          f"  mismatched {len(bad)}  unlisted {len(on_disk - listed)}  missing {len(listed - on_disk)}")
    return bad


# The three literature manifests keep their original "works" key; the unpacked folders
# and the papers use "files".
for name, key in [("oz", "works"), ("holmes", "works"), ("greek", "works"),
                  ("graphrag-bench", "files"), ("longmemeval", "files")]:
    check(RAW / name, key)
check(PAPERS, "files")


In [ ]:
# Block 2: file type, then raw text.
#
# Two steps, by bytes only. Nothing here decides what the text is about; that is the model's
# job later.
#   1. file_kind(data): look at the first bytes and name the container: pdf, json, or text.
#   2. to_text(path): turn the container into one string, the document text.
#        pdf  -> the text layer, page by page (PyMuPDF, the one dependency)
#        json -> if it holds chat turns, one "role: content" block per turn under a header
#                of the session id and dates. We also keep where each turn starts and ends
#                in that string, so a chat can be cut into units without a model.
#        text -> the bytes decoded as UTF-8, unchanged
import json

try:
    import pymupdf
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=True)
    import pymupdf


def file_kind(data):
    if data.startswith(b"%PDF-"):
        return "pdf"
    if data.lstrip()[:1] in (b"{", b"["):
        return "json"
    return "text"


def pdf_text(data):
    doc = pymupdf.open(stream=data, filetype="pdf")
    return "\n".join(page.get_text() for page in doc)


def chat_turns(obj):
    """The list of {role, content} turns inside a chat JSON, wherever it sits; None if absent."""
    if isinstance(obj, list) and obj and all(isinstance(t, dict) and "role" in t and "content" in t for t in obj):
        return obj
    if isinstance(obj, dict):
        for value in obj.values():
            found = chat_turns(value)
            if found:
                return found
    return None


def chat_text(obj, turns):
    """Header lines, a blank line, then 'role: content' per turn. Returns the text and the
    (start, end) of each turn inside it."""
    header = [f"session_id: {obj['session_id']}"] if "session_id" in obj else []
    header += [f"date: {d}" for d in obj.get("dates", [])]
    text = "\n".join(header) + "\n\n"
    spans = []
    for turn in turns:
        start = len(text)
        text += f"{turn['role']}: {turn['content']}\n\n"
        spans.append((start, len(text)))
    return text, spans


def to_text(path):
    data = path.read_bytes()
    doc = {"path": str(path), "sha256": hashlib.sha256(data).hexdigest(),
           "kind": file_kind(data), "text": "", "turns": None, "dates": []}
    if doc["kind"] == "pdf":
        doc["text"] = pdf_text(data)
    elif doc["kind"] == "json":
        obj = json.loads(data)
        turns = chat_turns(obj)
        if turns is None:
            doc["kind"], doc["text"] = "text", data.decode("utf-8", errors="replace")
        else:
            doc["kind"] = "chat"
            doc["text"], doc["turns"] = chat_text(obj, turns)
            doc["dates"] = list(obj.get("dates", []))
    else:
        doc["text"] = data.decode("utf-8", errors="replace")
    return doc


# One of each, to see the shape.
for path in [RAW / "oz" / "01_55.txt", RAW / "graphrag-bench" / "Novel-30752.txt",
             RAW / "longmemeval" / "sharegpt_yywfIrx_0.json", RAW / "longmemeval" / "001cefa7_2.json",
             PAPERS / "edge2024-graphrag.pdf"]:
    d = to_text(path)
    turns = len(d["turns"]) if d["turns"] else "-"
    print(f"{path.name:<26} {d['kind']:<5} {len(d['text']):>8,} chars  turns {turns:>3}  dates {d['dates']}")
    print("    " + repr(d["text"][:70]))


In [ ]:
# Block 3: the model interface.
# One JSON-mode call at a time, raw HTTP, every call logged with model, tokens, seconds,
# and cost. The spending stop is checked before each call. The matcher is the gate: a
# phrase the model returns must occur in the text, line wraps and spacing forgiven, and
# `locate` says where, searching forward from a position so a table of contents cannot
# steal a heading that appears again in the body.
import os
import re
import time

import requests

try:
    from kaggle_secrets import UserSecretsClient
    KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
except ImportError:
    KEY = os.environ.get("OPENAI_API_KEY", "")

MODEL = "gpt-5.6-luna"                # the split call
RETRY = "gpt-5.6-terra"               # one retry when the gates fail
PRICE = {"gpt-5.6-luna": (0.20, 1.20), "gpt-5.6-terra": (2.00, 12.00)}   # $ per M tokens in, out
SPEND_STOP = 2.00                     # dollars for the whole raw dataset; the run halts past this
calls = []


def spend():
    return sum(c["cost"] for c in calls)


def generate(prompt, stage, model=MODEL, effort="low"):
    """One JSON-mode call; the reply parsed, the cost logged. No temperature: the API
    rejects it; reasoning_effort is the only knob."""
    if spend() >= SPEND_STOP:
        raise RuntimeError(f"spending stop: ${spend():.2f}")
    t0 = time.time()
    r = requests.post("https://api.openai.com/v1/chat/completions",
                      headers={"Authorization": f"Bearer {KEY}"}, timeout=300,
                      json={"model": model, "reasoning_effort": effort,
                            "response_format": {"type": "json_object"},
                            "messages": [{"role": "user", "content": prompt}]})
    if r.status_code != 200:
        raise RuntimeError(f"OpenAI {r.status_code}: {r.text}")
    body = r.json()
    u, (p_in, p_out) = body["usage"], PRICE[model]
    calls.append({"stage": stage, "model": body["model"], "in": u["prompt_tokens"],
                  "out": u["completion_tokens"], "seconds": round(time.time() - t0, 1),
                  "cost": (u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out) / 1e6})
    return json.loads(body["choices"][0]["message"]["content"])


def pattern(s):
    return r"\s+".join(re.escape(w) for w in s.split())


def in_text(text, s):
    """The gate: the phrase occurs in the text, line wraps forgiven."""
    return bool(s.split()) and re.search(pattern(s), text) is not None


def locate(text, s, start=0):
    """(start, end) of the first occurrence of the phrase at or after `start`, else None."""
    if not s or not s.split():
        return None
    m = re.compile(pattern(s)).search(text, start)
    return (m.start(), m.end()) if m else None


print(f"model {MODEL}, retry {RETRY}, spend stop ${SPEND_STOP:.2f}, key {'present' if KEY else 'MISSING'}")


In [ ]:
# Block 4: the compressed view.
#
# The model never sees a whole book. It sees a numbered list of the places structure lives:
#   line view   the first and last 200 lines in full (title page, contents, the ending), and
#               in between only the short non-empty lines, because headings are short. "Short"
#               starts at 60 characters and tightens until the middle fits 3,000 entries, so a
#               verse play (every line short) still fits.
#   sentence view   for a text with almost no line breaks (one-line files), every sentence
#               start, first 60 characters each. A heading glued to the sentence after it
#               shows up as the start of that sentence.
# Every entry is verbatim text, so whatever the model quotes back can be found again.
import re

EDGE = 200
MAX_MIDDLE = 3000
PREFIX = 60
SENTENCE_START = re.compile(r'(?<=[.!?])\s+(?=[A-Z"“(\[])')


def compressed_view(text):
    if text.count("\n") < len(text) / 500:
        sentences = SENTENCE_START.split(text)
        entries = [(i, s[:PREFIX]) for i, s in enumerate(sentences)]
        mode, limit = "sentences", None
    else:
        lines = text.split("\n")
        first, last = range(min(EDGE, len(lines))), range(max(EDGE, len(lines) - EDGE), len(lines))
        for limit in (60, 40, 30, 20, 12):
            middle = [i for i in range(EDGE, len(lines) - EDGE) if lines[i].strip() and len(lines[i].rstrip("\r")) <= limit]
            if len(middle) <= MAX_MIDDLE:
                break
        entries = [(i, lines[i].rstrip("\r")[:120]) for i in [*first, *middle, *last]]
        mode = "lines"
    step = max(1, -(-len(entries) // (MAX_MIDDLE + 2 * EDGE)))    # ceiling division
    entries = entries[::step]
    body = "\n".join(f"{i}: {s}" for i, s in entries)
    return {"mode": mode, "limit": limit, "entries": len(entries), "sampled_every": step,
            "chars": len(body), "text": body}


for path in [RAW / "oz" / "01_55.txt", RAW / "greek" / "18_10523.txt", RAW / "graphrag-bench" / "Novel-30752.txt",
             PAPERS / "edge2024-graphrag.pdf"]:
    v = compressed_view(to_text(path)["text"])
    lines = v["text"].split("\n")
    print(f"{path.name:<26} {v['mode']:<9} short<={v['limit']}  entries {v['entries']:>5}  ~tokens {v['chars'] // 4:>6}"
          f"  sampled every {v['sampled_every']}")
    print("    " + "\n    ".join(lines[len(lines) // 2: len(lines) // 2 + 6]))


In [ ]:
# Block 5: the split call.
#
# One call per document. The model reads the compressed view and answers with JSON: what the
# document is, its title, author, translator, and date, where the work itself starts and
# ends (so front matter and end matter fall away), and the heading that opens each piece,
# with a short title for the piece. Every string it returns must be copied verbatim from the
# view, because the next block searches the document for it and cuts there. Nothing the
# model says is trusted until that search finds it.
CAP_WORDS = 4000

PROMPT = """Below is a compressed view of one document: numbered lines, or numbered sentence starts when the document has no line breaks. In the line view the first and last 200 lines are complete and the middle shows only short lines, which is where headings are.

Decide what the document is and how it divides into pieces. Answer with JSON only. Every string you return must be copied exactly from the view, character for character, never paraphrased or corrected, because a program will search the document for it.

{
  "source_class": one of "canonical" (a published literary or classic work), "published" (a paper, article, or report), "authored" (a person's own material: notes, email, letters, drafts),
  "title": the title as written, or null,
  "author": the author's name as written, or null,
  "date": {"quote": a verbatim phrase containing the date the work was written, published, or sent, "iso": "YYYY" or "YYYY-MM" or "YYYY-MM-DD"} or null. Not a transcription or ebook release date,
  "body_start": the verbatim opening of the first line of the work itself: the first piece's heading line when there are pieces, otherwise the first line of the text proper. Publisher notices, contents, and transcriber's or translator's notes are not part of the work; an author's own preface or introduction is,
  "body_end": the verbatim opening of the last line of the work itself, before any end matter (license, index, notes, advertisements),
  "toc_count": the number of pieces a table of contents lists, or null if there is none,
  "pieces": [{"marker": the verbatim opening of the heading line that begins the piece, in document order, "title": a short title for the piece, such as "Chapter 1: The Cyclone" or "Abstract" or "Act II, Scene 1"}]
}

Pieces are the document's own divisions: chapters, acts and scenes, sections, dated entries, poems, stories. A paper's pieces are its sections, the abstract first. Aim for pieces under %d words; where a division is longer, use its next level down. A document with no divisions gets an empty pieces list. Front matter and end matter are never pieces. Do not take markers from a table of contents; markers are the headings where each piece begins in the body.

VIEW:
%s
"""


def propose(doc, model=MODEL):
    """The model's proposal for one document, plus the view it saw."""
    view = compressed_view(doc["text"])
    reply = generate(PROMPT % (CAP_WORDS, view["text"]), stage="split", model=model)
    return reply, view


doc = to_text(RAW / "oz" / "01_55.txt")
reply, view = propose(doc)
for key in ("source_class", "title", "author", "date", "body_start", "body_end", "toc_count"):
    print(f"{key:<11} {reply.get(key)!r}")
print(f"pieces     {len(reply.get('pieces', []))}")
for piece in reply.get("pieces", [])[:5]:
    print(f"    {piece}")
print(f"call: {calls[-1]}")


In [ ]:
# Block 6: cut and verify.
#
# The model's reply is a proposal. This block turns it into offsets by searching the text:
#   1. body_start and body_end are found; everything outside them is boilerplate.
#   2. each marker is found in order, each search starting where the previous marker ended,
#      so a heading repeated in a contents list or a running head cannot be picked twice.
#      A marker anchors at a line start and must end at a word break, so "Chapter I" never
#      matches inside "Chapter II".
#   3. pieces run from one marker to the next; the last runs to body_end. Text between
#      body_start and the first marker, if any, is its own opening piece.
# Then the three gates from SCHEMA.md:
#   count      the number of pieces equals the table-of-contents count when the model saw one
#   coverage   pieces tile the body with no gap or overlap, and none holds a wildly
#              disproportionate share (the Metamorphoses failure: 97% in one piece)
#   round-trip joining the pieces reproduces the body text byte for byte
# Title, author, and the date quote are verified the same way; one that is not in the text
# is dropped, not trusted.


def find_marker(text, phrase, start, line_mode):
    """(start, end) of the phrase at or after start, anchored at a line start in line mode and
    ending at a word break; None when absent."""
    if not phrase or not phrase.split():
        return None
    core = pattern(phrase) + r"(?!\w)"
    m = re.compile((r"(?m)^[ \t]*" if line_mode else "") + core).search(text, start)
    return (m.start(), m.end()) if m else None


def find_last_before(text, phrase, pos, line_mode):
    """The last occurrence of the phrase starting at or before pos, else None. Used for the
    first heading: the model sometimes gives a body start inside the first piece rather
    than at its heading, and the heading nearest before it is the right one."""
    if not phrase or not phrase.split():
        return None
    core = (r"(?m)^[ \t]*" if line_mode else "") + pattern(phrase) + r"(?!\w)"
    hits = [(m.start(), m.end()) for m in re.compile(core).finditer(text, 0, pos + len(phrase) + 1) if m.start() <= pos]
    return hits[-1] if hits else None


def line_end(text, pos, line_mode):
    """End of the line (or, in a one-line document, of the sentence) containing pos."""
    if line_mode:
        nl = text.find("\n", pos)
        return len(text) if nl < 0 else nl + 1
    m = SENTENCE_START.search(text, pos)
    return len(text) if m is None else m.end()


def verified(text, value):
    """The string itself when it occurs in the text, else None."""
    return value if isinstance(value, str) and in_text(text, value) else None


def cut(doc, reply, view):
    text, line_mode = doc["text"], view["mode"] == "lines"
    plan = {"ok": False, "reason": None, "body": None, "pieces": [], "meta": {}, "notes": []}

    start = find_marker(text, reply.get("body_start"), 0, line_mode)
    if start is None:
        plan["reason"] = "body_start not found"
        return plan
    end = find_marker(text, reply.get("body_end"), start[0], line_mode)
    if end is None:
        plan["reason"] = "body_end not found"
        return plan
    body_start, body_end = start[0], line_end(text, end[1], line_mode)
    plan["body"] = (body_start, body_end)

    marks = []
    pos = body_start
    for i, piece in enumerate(reply.get("pieces") or []):
        if i == 0:
            hit = find_last_before(text, piece.get("marker"), body_start, line_mode) or \
                  find_marker(text, piece.get("marker"), body_start, line_mode)
        else:
            hit = find_marker(text, piece.get("marker"), pos, line_mode)
        if hit is None or hit[0] >= body_end:
            plan["reason"] = f"marker {i + 1} not found in order: {piece.get('marker')!r}"
            return plan
        marks.append((hit[0], piece.get("title") or piece.get("marker")))
        pos = hit[1]
    if marks and marks[0][0] < body_start:
        body_start = marks[0][0]            # the work starts at its first heading
        plan["body"] = (body_start, body_end)

    bounds = [m[0] for m in marks] + [body_end]
    pieces = []
    if marks and text[body_start:marks[0][0]].strip():
        pieces.append({"start": body_start, "end": marks[0][0], "label": "opening"})
    for (s, label), e in zip(marks, bounds[1:]):
        pieces.append({"start": s, "end": e, "label": label})
    if not marks:
        pieces.append({"start": body_start, "end": body_end, "label": reply.get("title") or "whole"})
    plan["pieces"] = pieces

    # Gate 1: count.
    toc = reply.get("toc_count")
    if isinstance(toc, int) and toc > 0 and len(marks) != toc:
        plan["reason"] = f"count: {len(marks)} pieces, contents lists {toc}"
        return plan
    # Gate 2: coverage.
    tiled = pieces[0]["start"] == body_start and pieces[-1]["end"] == body_end and all(
        a["end"] == b["start"] for a, b in zip(pieces, pieces[1:]))
    if not tiled:
        plan["reason"] = "coverage: pieces do not tile the body"
        return plan
    share = max(p["end"] - p["start"] for p in pieces) / max(1, body_end - body_start)
    limit = max(3 / len(pieces), 0.10)
    if len(pieces) > 1 and share > limit:
        plan["reason"] = f"coverage: one piece holds {share:.0%} of the body (limit {limit:.0%})"
        return plan
    # Gate 3: round-trip.
    if "".join(text[p["start"]:p["end"]] for p in pieces) != text[body_start:body_end]:
        plan["reason"] = "round-trip: joined pieces differ from the body"
        return plan

    plan["meta"]["source_class"] = reply.get("source_class") if reply.get("source_class") in ("canonical", "published", "authored") else None
    if plan["meta"]["source_class"] is None:
        plan["notes"].append(f"source_class not one of the three: {reply.get('source_class')!r}")
    for key in ("title", "author"):
        plan["meta"][key] = verified(text, reply.get(key))
        if reply.get(key) and plan["meta"][key] is None:
            plan["notes"].append(f"{key} not found in text: {reply.get(key)!r}")
    date = reply.get("date") or {}
    plan["meta"]["date"] = {"quote": date["quote"], "iso": date.get("iso")} if verified(text, date.get("quote")) else None
    if date and plan["meta"]["date"] is None:
        plan["notes"].append(f"date quote not found in text: {date!r}")
    plan["boilerplate_share"] = 1 - (body_end - body_start) / max(1, len(text))
    plan["ok"] = True
    return plan


plan = cut(doc, reply, view)
print("ok" if plan["ok"] else f"FAILED: {plan['reason']}", "| pieces", len(plan["pieces"]),
      "| body", plan["body"], f"| boilerplate {plan.get('boilerplate_share', 0):.1%}")
print("meta:", plan["meta"])
for note in plan["notes"]:
    print("note:", note)
for p in plan["pieces"][:3] + plan["pieces"][-2:]:
    words = len(doc["text"][p["start"]:p["end"]].split())
    print(f"  {p['label']:<40} {words:>6} words  {doc['text'][p['start']:p['start'] + 50]!r}")
